# 04 - Gobernabilidad y Seguridad de Datos

**Proyecto:** Plataforma Big Data para Analítica Omnicanal de Ventas Retail  
**Equipo:** Ballerini · Torres · Vargas · Vásquez  
**Fase:** 2 — Implementación y Optimización  

---

Este notebook implementa los controles de gobierno y seguridad exigidos por la **Fase 2** del enunciado:

| # | Control | Herramienta GCP |
|---|---------|-----------------|
| 1 | Control de acceso (IAM) | BigQuery IAM + Dataset ACLs |
| 2 | Linaje de datos | `retail_governance.data_lineage` |
| 3 | Manejo de datos sensibles | Pseudonimización SHA-256 + Policy Tags |
| 4 | Auditoría de pipeline | `retail_governance.pipeline_audit_log` |
| 5 | Calidad de datos | Great Expectations + `data_quality_results` |

> **Declaración de IA:** Este notebook fue asistido por Claude (Anthropic) para la redacción de comentarios explicativos. Las decisiones arquitectónicas son propias del equipo.


## 0. Instalación de Dependencias

In [1]:
# 1. Instalar dependencias requeridas
!pip install -q \
    google-cloud-bigquery \
    google-cloud-storage \
    great_expectations==0.18.15 \
    db-dtypes

# 2. Forzar la actualización de NumPy y Pandas para corregir el conflicto binario en Python 3.12
!pip install -q --upgrade numpy pandas

print("Dependencias actualizadas y corregidas con éxito.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92

## 1. Autenticación y Configuración

In [2]:
from google.colab import auth
auth.authenticate_user()
print("Autenticacion GCP exitosa.")


Autenticacion GCP exitosa.


In [3]:
# ── PARAMETROS GLOBALES ─────────────────────────────────────
# Ajustar PROJECT_ID al proyecto GCP real del equipo.
PROJECT_ID   = "proyectointegradorudd"   # <-- CAMBIAR por tu Project ID
REGION       = "us-central1"
BUCKET_NAME  = "data-lake-retail"
DATASET_BQ   = "retail_gold"
DATASET_GOV  = "retail_governance"    # Dataset de gobierno

print(f"Proyecto : {PROJECT_ID}")
print(f"Dataset  : {PROJECT_ID}.{DATASET_BQ}")
print(f"Gobierno : {PROJECT_ID}.{DATASET_GOV}")


Proyecto : proyectointegradorudd
Dataset  : proyectointegradorudd.retail_gold
Gobierno : proyectointegradorudd.retail_governance


## 2. Control de Acceso — IAM y Dataset ACLs

Estrategia de **mínimo privilegio**: cada rol de negocio recibe solo los permisos necesarios.

| Rol de negocio | Cuenta de servicio | Permiso BigQuery |
|---|---|---|
| Data Engineer | `svc-dataeng@PROJECT.iam` | `WRITER` en todas las capas |
| Analista Comercial | `svc-analyst@PROJECT.iam` | `READER` solo en Gold |
| Orquestador (Airflow) | `svc-airflow@PROJECT.iam` | `WRITER` + `jobUser` |
| Auditor | `svc-auditor@PROJECT.iam` | `READER` solo en Governance |


In [4]:
from google.cloud import bigquery
from google.cloud.exceptions import NotFound

bq = bigquery.Client(project=PROJECT_ID)

# ── DEFINICIÓN DE ACLS POR DATASET ──────────────────────────
ACCESS_POLICIES = {
    DATASET_BQ: [
        bigquery.AccessEntry(
            role="WRITER",
            entity_type="serviceAccount",
            entity_id=f"svc-dataeng@{PROJECT_ID}.iam.gserviceaccount.com"
        ),
        bigquery.AccessEntry(
            role="READER",
            entity_type="serviceAccount",
            entity_id=f"svc-analyst@{PROJECT_ID}.iam.gserviceaccount.com"
        ),
        bigquery.AccessEntry(
            role="WRITER",
            entity_type="serviceAccount",
            entity_id=f"svc-airflow@{PROJECT_ID}.iam.gserviceaccount.com"
        ),
    ],
    DATASET_GOV: [
        bigquery.AccessEntry(
            role="WRITER",
            entity_type="serviceAccount",
            entity_id=f"svc-dataeng@{PROJECT_ID}.iam.gserviceaccount.com"
        ),
        bigquery.AccessEntry(
            role="READER",
            entity_type="serviceAccount",
            entity_id=f"svc-auditor@{PROJECT_ID}.iam.gserviceaccount.com"
        ),
    ]
}

def aplicar_politicas_acceso(dataset_id, nuevas_entradas):
    """
    Crea el dataset si no existe y aplica las políticas de acceso correspondientes.
    """
    ref = f"{PROJECT_ID}.{dataset_id}"
    
    # 1. Asegurar la existencia del dataset primero
    try:
        dataset = bq.get_dataset(ref)
    except NotFound:
        print(f"INFO: El dataset {ref} no existe. Creándolo ahora...")
        ds = bigquery.Dataset(ref)
        ds.location = REGION
        dataset = bq.create_dataset(ds)
        print(f"OK: Dataset creado exitosamente: {ref}")

    # 2. Aplicar los accesos (ACLs)
    try:
        entradas_actuales   = list(dataset.access_entries)
        entity_ids_actuales = {e.entity_id for e in entradas_actuales if e.entity_id}
        
        entradas_nuevas = [
            e for e in nuevas_entradas
            if e.entity_id not in entity_ids_actuales
        ]
        
        if entradas_nuevas:
            dataset.access_entries = entradas_actuales + entradas_nuevas
            bq.update_dataset(dataset, ["access_entries"])
            print(f"OK: Políticas aplicadas en: {dataset_id}")
            for e in entradas_nuevas:
                print(f"    + [{e.role}] -> {e.entity_id}")
        else:
            print(f"INFO: Todas las entradas ya estaban configuradas en {dataset_id}.")
            
    except Exception as ex:
        print(f"AVISO: No se pudieron actualizar los accesos de {dataset_id}: {ex}")

# Ejecutar el proceso para ambos datasets
for dataset_id, entradas in ACCESS_POLICIES.items():
    aplicar_politicas_acceso(dataset_id, entradas)

INFO: El dataset proyectointegradorudd.retail_gold no existe. Creándolo ahora...
OK: Dataset creado exitosamente: proyectointegradorudd.retail_gold
AVISO: No se pudieron actualizar los accesos de retail_gold: 400 PATCH https://bigquery.googleapis.com/bigquery/v2/projects/proyectointegradorudd/datasets/retail_gold?prettyPrint=false: An access entry must have exactly one of userByEmail, groupByEmail, domain, specialGroup defined, view, routine, or dataset.
INFO: El dataset proyectointegradorudd.retail_governance no existe. Creándolo ahora...
OK: Dataset creado exitosamente: proyectointegradorudd.retail_governance
AVISO: No se pudieron actualizar los accesos de retail_governance: 400 PATCH https://bigquery.googleapis.com/bigquery/v2/projects/proyectointegradorudd/datasets/retail_governance?prettyPrint=false: An access entry must have exactly one of userByEmail, groupByEmail, domain, specialGroup defined, view, routine, or dataset.


In [5]:
# ── Verificacion de politicas aplicadas ─────────────────────
def mostrar_politicas_dataset(dataset_id):
    try:
        dataset = bq.get_dataset(dataset_id)
        print(f"\nPOLITICAS DE ACCESO: {dataset_id}")
        print("-"*60)
        print(f"{'ROL':<15} {'TIPO':<22} {'PRINCIPAL'}")
        print("-"*60)
        for e in dataset.access_entries:
            rol    = e.role or "SPECIAL"
            tipo   = e.entity_type or "—"
            entid  = e.entity_id or str(e.special_group or "")
            print(f"{rol:<15} {tipo:<22} {entid}")
    except Exception as ex:
        print(f"AVISO Error al leer {dataset_id}: {ex}")

mostrar_politicas_dataset(DATASET_BQ)
mostrar_politicas_dataset(DATASET_GOV)



POLITICAS DE ACCESO: retail_gold
------------------------------------------------------------
ROL             TIPO                   PRINCIPAL
------------------------------------------------------------
WRITER          specialGroup           projectWriters
OWNER           specialGroup           projectOwners
OWNER           userByEmail            cballerinip@gmail.com
READER          specialGroup           projectReaders

POLITICAS DE ACCESO: retail_governance
------------------------------------------------------------
ROL             TIPO                   PRINCIPAL
------------------------------------------------------------
WRITER          specialGroup           projectWriters
OWNER           specialGroup           projectOwners
OWNER           userByEmail            cballerinip@gmail.com
READER          specialGroup           projectReaders


## 3. Linaje de Datos

Documentamos el flujo de cada campo desde RAW hasta Gold en la tabla
`retail_governance.data_lineage`. Estrategia **registro manual estructurado**,
válida según el enunciado: *"documentación del linaje de datos (puede ser manual)"*.

El linaje responde tres preguntas clave de auditoría:
1. ¿De dónde viene este campo?
2. ¿Qué transformaciones se le aplicaron?
3. ¿Es un dato sensible y cómo está protegido?


In [6]:
# ── Crear dataset de gobierno si no existe ───────────────────
def crear_dataset_si_no_existe(dataset_id, descripcion=""):
    ref = f"{PROJECT_ID}.{dataset_id}"
    try:
        bq.get_dataset(ref)
        print(f"INFO Dataset ya existe: {ref}")
    except Exception:
        ds = bigquery.Dataset(ref)
        ds.location    = REGION
        ds.description = descripcion
        bq.create_dataset(ds)
        print(f"OK   Dataset creado: {ref}")

crear_dataset_si_no_existe(
    DATASET_GOV,
    "Dataset de gobierno: linaje, auditoria y calidad de datos."
)


INFO Dataset ya existe: proyectointegradorudd.retail_governance


In [7]:
# ── Crear tabla de linaje ────────────────────────────────────
SCHEMA_LINAJE = [
    bigquery.SchemaField("id_linaje",          "STRING",    "REQUIRED",
        description="UUID unico del registro"),
    bigquery.SchemaField("nombre_campo",       "STRING",    "REQUIRED",
        description="Nombre de la columna documentada"),
    bigquery.SchemaField("tabla_origen",       "STRING",    "REQUIRED",
        description="Ruta o tabla de origen"),
    bigquery.SchemaField("tabla_destino",      "STRING",    "REQUIRED",
        description="Ruta o tabla destino"),
    bigquery.SchemaField("capa_origen",        "STRING",    "REQUIRED",
        description="Capa Medallion: RAW | BRONZE | SILVER | GOLD"),
    bigquery.SchemaField("capa_destino",       "STRING",    "REQUIRED",
        description="Capa Medallion: BRONZE | SILVER | GOLD"),
    bigquery.SchemaField("transformacion",     "STRING",    "NULLABLE",
        description="Descripcion de la transformacion aplicada"),
    bigquery.SchemaField("notebook_origen",    "STRING",    "NULLABLE",
        description="Notebook o script que ejecuto la transformacion"),
    bigquery.SchemaField("es_dato_sensible",   "BOOL",      "REQUIRED",
        description="TRUE si contiene PII u otro dato sensible"),
    bigquery.SchemaField("tecnica_proteccion", "STRING",    "NULLABLE",
        description="HASH | MASK | SUPPRESS | NONE"),
    bigquery.SchemaField("fecha_registro",     "TIMESTAMP", "REQUIRED"),
    bigquery.SchemaField("responsable",        "STRING",    "REQUIRED"),
]

tabla_linaje_ref = f"{PROJECT_ID}.{DATASET_GOV}.data_lineage"

def crear_tabla_si_no_existe(tabla_ref, schema, descripcion=""):
    try:
        bq.get_table(tabla_ref)
        print(f"INFO Tabla ya existe: {tabla_ref}")
    except Exception:
        t = bigquery.Table(tabla_ref, schema=schema)
        t.description = descripcion
        bq.create_table(t)
        print(f"OK   Tabla creada: {tabla_ref}")

crear_tabla_si_no_existe(tabla_linaje_ref, SCHEMA_LINAJE,
    "Linaje de datos del pipeline retail. Una fila por campo transformado.")


OK   Tabla creada: proyectointegradorudd.retail_governance.data_lineage


In [8]:
# ── Poblar linaje del pipeline completo ─────────────────────
import uuid
from datetime import datetime, timezone

RESPONSABLE = f"svc-dataeng@{PROJECT_ID}.iam.gserviceaccount.com"
TS_AHORA    = datetime.now(timezone.utc).isoformat()
B  = f"gs://{BUCKET_NAME}/bronze/venta_tiendas_delta"
S  = f"gs://{BUCKET_NAME}/silver/venta_tiendas_delta"
G  = f"{PROJECT_ID}.{DATASET_BQ}.mart_ventas_mensual"
RAW = f"gs://{BUCKET_NAME}/raw/venta_tiendas.csv"

# (campo, origen, destino, c_org, c_dst, transformacion, notebook, sensible, proteccion)
REGISTROS_LINAJE = [
    # RAW -> BRONZE
    ("numero_transaccion", RAW, B, "RAW","BRONZE",
     "Copia directa. Bronze preserva el dato crudo sin transformacion.",
     "01_Capa_Bronze.ipynb", False, "NONE"),
    ("fecha_transaccion", RAW, B, "RAW","BRONZE",
     "Copia directa. Formato original: dd/MM/yyyy hh:mm:ss a CL.",
     "01_Capa_Bronze.ipynb", False, "NONE"),
    ("venta", RAW, B, "RAW","BRONZE",
     "Copia directa como STRING. El cast a DOUBLE ocurre en Silver.",
     "01_Capa_Bronze.ipynb", False, "NONE"),
    ("id_producto", RAW, B, "RAW","BRONZE",
     "Copia directa. Tipo LONG asignado en Silver.",
     "01_Capa_Bronze.ipynb", False, "NONE"),
    # BRONZE -> SILVER
    ("fecha_venta", B, S, "BRONZE","SILVER",
     "Creado con to_date(to_timestamp(fecha_transaccion)). "
     "Sufijo ' CL' eliminado con regexp_replace.",
     "02_Capa_Silver.ipynb", False, "NONE"),
    ("margen", B, S, "BRONZE","SILVER",
     "Campo derivado: venta - costo. Margen bruto por transaccion.",
     "02_Capa_Silver.ipynb", False, "NONE"),
    ("margen_porcentaje", B, S, "BRONZE","SILVER",
     "(venta - costo) / venta cuando venta > 0, NULL en caso contrario.",
     "02_Capa_Silver.ipynb", False, "NONE"),
    ("venta", B, S, "BRONZE","SILVER",
     "Cast a DOUBLE. Se excluyen registros con venta NULL.",
     "02_Capa_Silver.ipynb", False, "NONE"),
    # SILVER -> GOLD
    ("venta_total", S, G, "SILVER","GOLD",
     "SUM(venta) GROUP BY anio, mes, cod_tienda_facturacion.",
     "03_Capa_Gold_v2.ipynb", False, "NONE"),
    ("margen_total", S, G, "SILVER","GOLD",
     "SUM(margen) GROUP BY anio, mes, cod_tienda_facturacion.",
     "03_Capa_Gold_v2.ipynb", False, "NONE"),
    ("numero_transacciones", S, G, "SILVER","GOLD",
     "COUNT(numero_transaccion) GROUP BY anio, mes, cod_tienda.",
     "03_Capa_Gold_v2.ipynb", False, "NONE"),
]

filas_bq = [{
    "id_linaje":          str(uuid.uuid4()),
    "nombre_campo":       c, "tabla_origen": o, "tabla_destino": d,
    "capa_origen":        co, "capa_destino": cd,
    "transformacion":     t, "notebook_origen": nb,
    "es_dato_sensible":   s, "tecnica_proteccion": p,
    "fecha_registro":     TS_AHORA, "responsable": RESPONSABLE,
} for c,o,d,co,cd,t,nb,s,p in REGISTROS_LINAJE]

errores = bq.insert_rows_json(tabla_linaje_ref, filas_bq)
if errores:
    print("AVISO Errores al insertar linaje:", errores)
else:
    print(f"OK   Linaje registrado: {len(filas_bq)} campos documentados")


OK   Linaje registrado: 11 campos documentados


In [9]:
# ── Consultar y visualizar el linaje ────────────────────────
query_linaje = f"""
SELECT
    capa_origen,
    capa_destino,
    nombre_campo,
    es_dato_sensible,
    tecnica_proteccion,
    notebook_origen,
    SUBSTR(transformacion, 1, 65) AS transformacion_resumen
FROM `{PROJECT_ID}.{DATASET_GOV}.data_lineage`
ORDER BY
    CASE capa_origen WHEN 'RAW' THEN 1 WHEN 'BRONZE' THEN 2
                     WHEN 'SILVER' THEN 3 ELSE 4 END,
    nombre_campo
"""

import pandas as pd
df_linaje = bq.query(query_linaje).to_dataframe()
print("\nREGISTRO DE LINAJE DEL PIPELINE")
print("="*80)
print(df_linaje.to_string(index=False, max_colwidth=65))
print(f"\nTotal de campos documentados: {len(df_linaje)}")



REGISTRO DE LINAJE DEL PIPELINE
capa_origen capa_destino         nombre_campo  es_dato_sensible tecnica_proteccion       notebook_origen                                            transformacion_resumen
        RAW       BRONZE    fecha_transaccion             False               NONE  01_Capa_Bronze.ipynb        Copia directa. Formato original: dd/MM/yyyy hh:mm:ss a CL.
        RAW       BRONZE          id_producto             False               NONE  01_Capa_Bronze.ipynb                      Copia directa. Tipo LONG asignado en Silver.
        RAW       BRONZE   numero_transaccion             False               NONE  01_Capa_Bronze.ipynb  Copia directa. Bronze preserva el dato crudo sin transformacion.
        RAW       BRONZE                venta             False               NONE  01_Capa_Bronze.ipynb     Copia directa como STRING. El cast a DOUBLE ocurre en Silver.
     BRONZE       SILVER          fecha_venta             False               NONE  02_Capa_Silver.ipynb Creado 

## 4. Manejo de Datos Sensibles

El dataset de ventas no contiene PII en su forma actual (sin RUT, nombre ni email).
Sin embargo, implementamos controles preventivos para cuando el sistema se extienda
con datos de clientes.

### Estrategia de tres capas

| Tipo de dato | Técnica | Justificacion |
|---|---|---|
| Identificadores de cliente | **SHA-256** (pseudonimizacion) | Permite joins sin exponer identidad |
| Montos financieros individuales | **Enmascaramiento a rangos** | Analistas ven rangos, no montos exactos |
| Columnas de alto riesgo | **Column-level Security (Policy Tags)** | Acceso granular en BigQuery |


In [10]:
# ── 4A. Pseudonimizacion con SHA-256 ─────────────────────────
import hashlib

def pseudonimizar(valor, salt="retail_udd_2026"):
    """
    Aplica SHA-256 con salt al valor.
    
    - Determinista: mismo input -> mismo hash.
    - No reversible sin conocer el salt.
    - Compatible con joins entre tablas.
    
    En produccion: cargar el salt desde Secret Manager,
    no hardcodearlo en el codigo.
    """
    if valor is None:
        return None
    entrada = f"{valor}{salt}".encode("utf-8")
    return hashlib.sha256(entrada).hexdigest()


# Demostracion con datos de ejemplo
datos_ejemplo = [
    {"id_cliente": "12345678-9", "nombre": "Juan Perez",    "monto": 45000},
    {"id_cliente": "98765432-1", "nombre": "Maria Gonzalez","monto": 12500},
    {"id_cliente": "11111111-1", "nombre": "Carlos Lopez",  "monto": 89000},
]

print("ANTES de pseudonimizar:")
print(f"{'ID Cliente':<20} {'Nombre':<20} {'Monto':>10}")
print("-"*52)
for d in datos_ejemplo:
    print(f"{d['id_cliente']:<20} {d['nombre']:<20} {d['monto']:>10,}")

print("\nDESPUES de pseudonimizar:")
print(f"{'ID_hash (primeros 20 chars)':<30} {'Nombre':<10} {'Monto':>10}")
print("-"*52)
for d in datos_ejemplo:
    hash_id = pseudonimizar(d['id_cliente'])
    print(f"{hash_id[:20]}...  {'[OCULTO]':<10} {d['monto']:>10,}")

print("\nOK  Datos personales no recuperables sin el salt.")


ANTES de pseudonimizar:
ID Cliente           Nombre                    Monto
----------------------------------------------------
12345678-9           Juan Perez               45,000
98765432-1           Maria Gonzalez           12,500
11111111-1           Carlos Lopez             89,000

DESPUES de pseudonimizar:
ID_hash (primeros 20 chars)    Nombre          Monto
----------------------------------------------------
4d9c42a961ccb9af1b36...  [OCULTO]       45,000
971517f4ccb40e0b5b46...  [OCULTO]       12,500
55f97c32c4b968d00d1d...  [OCULTO]       89,000

OK  Datos personales no recuperables sin el salt.


In [11]:
# ── 4B. Enmascaramiento de montos financieros ────────────────
def asignar_rango_venta(monto):
    """
    Transforma un monto exacto en un rango categorico.
    En Gold, los analistas ven rangos, no montos por boleta.
    
    Rangos en CLP:
        DEVOLUCION_O_CERO : monto <= 0
        0-10k             : menos de 10.000
        10k-50k           : entre 10.000 y 49.999
        50k-100k          : entre 50.000 y 99.999
        100k+             : 100.000 o mas
    """
    if monto is None:
        return "DESCONOCIDO"
    if monto <= 0:
        return "DEVOLUCION_O_CERO"
    elif monto < 10_000:
        return "0-10k"
    elif monto < 50_000:
        return "10k-50k"
    elif monto < 100_000:
        return "50k-100k"
    else:
        return "100k+"

montos_prueba = [-5000, 0, 3200, 25000, 67000, 150000, None]
print("ENMASCARAMIENTO DE MONTOS FINANCIEROS")
print(f"{'Monto original':>20}  ->  Rango Gold")
print("-"*40)
for m in montos_prueba:
    print(f"{str(m):>20}  ->  {asignar_rango_venta(m)}")


ENMASCARAMIENTO DE MONTOS FINANCIEROS
      Monto original  ->  Rango Gold
----------------------------------------
               -5000  ->  DEVOLUCION_O_CERO
                   0  ->  DEVOLUCION_O_CERO
                3200  ->  0-10k
               25000  ->  10k-50k
               67000  ->  50k-100k
              150000  ->  100k+
                None  ->  DESCONOCIDO


In [12]:
# ── 4C. Column-level Security — Policy Tags ──────────────────
# BigQuery aplica control de acceso a nivel de columna via Policy Tags.
# Usuarios sin el rol Fine-Grained Reader reciben NULL en esas columnas.
# 
# La creacion de taxonomias de Policy Tags requiere Data Catalog Admin.
# Aqui documentamos la configuracion y el procedimiento de aplicacion.

CLASIFICACION_COLUMNAS = {
    "CONFIDENCIAL": {
        "descripcion":    "Dato financiero individual. Requiere rol analyst-financial.",
        "columnas":       ["numero_boleta", "id_cliente_hash"],
        "rol_requerido":  "roles/datacatalog.categoryFineGrainedReader"
    },
    "INTERNO": {
        "descripcion":    "Dato comercial sensible. Uso restringido a ingenieria.",
        "columnas":       ["costo", "margen_porcentaje"],
        "rol_requerido":  "roles/datacatalog.categoryFineGrainedReader"
    },
    "PUBLICO": {
        "descripcion":    "Disponible para todos los lectores del dataset.",
        "columnas":       ["fecha_venta","id_canal","id_producto","unidades","anio","mes"],
        "rol_requerido":  None
    }
}

print("CLASIFICACION DE COLUMNAS POR NIVEL DE SENSIBILIDAD")
print("="*65)
for nivel, cfg in CLASIFICACION_COLUMNAS.items():
    print(f"\nNivel: {nivel}")
    print(f"  Descripcion  : {cfg['descripcion']}")
    print(f"  Rol requerido: {cfg['rol_requerido'] or 'Sin restriccion'}")
    print(f"  Columnas     : {', '.join(cfg['columnas'])}")

print("\nPROCEDIMIENTO PARA APLICAR POLICY TAGS EN CONSOLA GCP:")
print("  1. BigQuery Studio > tu_dataset > tu_tabla > Schema")
print("  2. Clic en la columna a proteger > Edit > Policy tag")
print("  3. Seleccionar la taxonomia y el tag apropiado")
print("  4. Guardar — BigQuery aplica el control de forma inmediata")


CLASIFICACION DE COLUMNAS POR NIVEL DE SENSIBILIDAD

Nivel: CONFIDENCIAL
  Descripcion  : Dato financiero individual. Requiere rol analyst-financial.
  Rol requerido: roles/datacatalog.categoryFineGrainedReader
  Columnas     : numero_boleta, id_cliente_hash

Nivel: INTERNO
  Descripcion  : Dato comercial sensible. Uso restringido a ingenieria.
  Rol requerido: roles/datacatalog.categoryFineGrainedReader
  Columnas     : costo, margen_porcentaje

Nivel: PUBLICO
  Descripcion  : Disponible para todos los lectores del dataset.
  Rol requerido: Sin restriccion
  Columnas     : fecha_venta, id_canal, id_producto, unidades, anio, mes

PROCEDIMIENTO PARA APLICAR POLICY TAGS EN CONSOLA GCP:
  1. BigQuery Studio > tu_dataset > tu_tabla > Schema
  2. Clic en la columna a proteger > Edit > Policy tag
  3. Seleccionar la taxonomia y el tag apropiado
  4. Guardar — BigQuery aplica el control de forma inmediata


## 5. Registro de Auditoría del Pipeline

Toda ejecucion del pipeline queda trazada en `retail_governance.pipeline_audit_log`.
La clase `AuditorPipeline` es reutilizable en los notebooks Bronze, Silver y Gold.


In [13]:
# ── Crear tabla de auditoria ─────────────────────────────────
SCHEMA_AUDITORIA = [
    bigquery.SchemaField("id_ejecucion",         "STRING",    "REQUIRED"),
    bigquery.SchemaField("etapa",                "STRING",    "REQUIRED",
        description="BRONZE | SILVER | GOLD | GOVERNANCE"),
    bigquery.SchemaField("notebook",             "STRING",    "REQUIRED"),
    bigquery.SchemaField("estado",               "STRING",    "REQUIRED",
        description="INICIO | EXITO | ERROR"),
    bigquery.SchemaField("registros_entrada",    "INTEGER",   "NULLABLE"),
    bigquery.SchemaField("registros_salida",     "INTEGER",   "NULLABLE"),
    bigquery.SchemaField("registros_descartados","INTEGER",   "NULLABLE"),
    bigquery.SchemaField("duracion_segundos",    "FLOAT64",   "NULLABLE"),
    bigquery.SchemaField("mensaje_error",        "STRING",    "NULLABLE"),
    bigquery.SchemaField("cuenta_ejecucion",     "STRING",    "REQUIRED"),
    bigquery.SchemaField("timestamp_inicio",     "TIMESTAMP", "REQUIRED"),
    bigquery.SchemaField("timestamp_fin",        "TIMESTAMP", "NULLABLE"),
    bigquery.SchemaField("version_codigo",       "STRING",    "NULLABLE",
        description="Git commit hash para trazabilidad"),
]

tabla_auditoria_ref = f"{PROJECT_ID}.{DATASET_GOV}.pipeline_audit_log"
crear_tabla_si_no_existe(tabla_auditoria_ref, SCHEMA_AUDITORIA,
    "Log de auditoria del pipeline ETL retail. Un registro por etapa ejecutada.")


OK   Tabla creada: proyectointegradorudd.retail_governance.pipeline_audit_log


In [14]:
# ── Clase AuditorPipeline ────────────────────────────────────
import time as _time

class AuditorPipeline:
    """
    Registra el inicio y fin de cada etapa del pipeline en BigQuery.
    
    Uso en cualquier notebook del pipeline:
    
        auditor = AuditorPipeline("SILVER", "02_Capa_Silver.ipynb")
        auditor.inicio()
        # ... procesamiento Spark ...
        auditor.fin(registros_entrada=12_000_000,
                    registros_salida=11_987_340,
                    registros_descartados=12_660,
                    duracion_seg=45.7)
    """
    
    def __init__(self, etapa, notebook, cuenta=None, version_codigo="HEAD"):
        self.id_ejecucion   = str(uuid.uuid4())
        self.etapa          = etapa
        self.notebook       = notebook
        self.cuenta         = cuenta or RESPONSABLE
        self.version_codigo = version_codigo
        self._ts_inicio     = None

    def inicio(self):
        self._ts_inicio = datetime.now(timezone.utc)
        fila = [{
            "id_ejecucion": self.id_ejecucion, "etapa": self.etapa,
            "notebook": self.notebook, "estado": "INICIO",
            "registros_entrada": None, "registros_salida": None,
            "registros_descartados": None, "duracion_segundos": None,
            "mensaje_error": None, "cuenta_ejecucion": self.cuenta,
            "timestamp_inicio": self._ts_inicio.isoformat(),
            "timestamp_fin": None, "version_codigo": self.version_codigo,
        }]
        errores = bq.insert_rows_json(tabla_auditoria_ref, fila)
        if not errores:
            print(f"[{self.etapa}] Inicio registrado — ID: {self.id_ejecucion[:8]}...")
        return self

    def fin(self, registros_entrada=None, registros_salida=None,
            registros_descartados=None, duracion_seg=None, error=None):
        ts_fin = datetime.now(timezone.utc)
        estado = "ERROR" if error else "EXITO"
        fila = [{
            "id_ejecucion": self.id_ejecucion, "etapa": self.etapa,
            "notebook": self.notebook, "estado": estado,
            "registros_entrada": registros_entrada,
            "registros_salida":  registros_salida,
            "registros_descartados": registros_descartados,
            "duracion_segundos": duracion_seg,
            "mensaje_error":     str(error) if error else None,
            "cuenta_ejecucion":  self.cuenta,
            "timestamp_inicio":  self._ts_inicio.isoformat() if self._ts_inicio else ts_fin.isoformat(),
            "timestamp_fin":     ts_fin.isoformat(),
            "version_codigo":    self.version_codigo,
        }]
        errores = bq.insert_rows_json(tabla_auditoria_ref, fila)
        icono = "OK" if estado == "EXITO" else "ERROR"
        if not errores:
            entrada   = f"{registros_entrada:,}" if registros_entrada else "—"
            salida    = f"{registros_salida:,}"  if registros_salida  else "—"
            descart   = f"{registros_descartados:,}" if registros_descartados else "—"
            dur       = f"{duracion_seg:.1f}s" if duracion_seg else "—"
            print(f"[{self.etapa}] {icono} | Entrada: {entrada} | "
                  f"Salida: {salida} | Descartados: {descart} | Tiempo: {dur}")


In [15]:
# ── Simulacion de ejecucion del pipeline completo ────────────
print("SIMULACION DE EJECUCIONES DEL PIPELINE\n")

ejecuciones = [
    ("BRONZE",     "01_Capa_Bronze.ipynb",   12_000_000, 12_000_000, 0,      18.4),
    ("SILVER",     "02_Capa_Silver.ipynb",   12_000_000, 11_987_340, 12_660, 45.7),
    ("GOLD",       "03_Capa_Gold_v2.ipynb",  11_987_340,      3_650, 0,      12.1),
    ("GOVERNANCE", "04_Gobernabilidad.ipynb",         0,          0, 0,       3.2),
]

for etapa, notebook, entrada, salida, descartados, duracion in ejecuciones:
    aud = AuditorPipeline(etapa, notebook)
    aud.inicio()
    _time.sleep(0.05)
    aud.fin(
        registros_entrada=entrada,
        registros_salida=salida,
        registros_descartados=descartados,
        duracion_seg=duracion
    )


SIMULACION DE EJECUCIONES DEL PIPELINE

[BRONZE] Inicio registrado — ID: 5e3e8f29...
[BRONZE] OK | Entrada: 12,000,000 | Salida: 12,000,000 | Descartados: — | Tiempo: 18.4s
[SILVER] Inicio registrado — ID: 5aba44bc...
[SILVER] OK | Entrada: 12,000,000 | Salida: 11,987,340 | Descartados: 12,660 | Tiempo: 45.7s
[GOLD] Inicio registrado — ID: 01e8d834...
[GOLD] OK | Entrada: 11,987,340 | Salida: 3,650 | Descartados: — | Tiempo: 12.1s
[GOVERNANCE] Inicio registrado — ID: 23da1616...
[GOVERNANCE] OK | Entrada: — | Salida: — | Descartados: — | Tiempo: 3.2s


In [16]:
# ── Consulta de resumen de auditoria ────────────────────────
query_audit = f"""
SELECT
    etapa,
    estado,
    FORMAT_TIMESTAMP('%Y-%m-%d %H:%M', timestamp_inicio) AS ejecutado_en,
    registros_entrada,
    registros_salida,
    registros_descartados,
    ROUND(duracion_segundos, 1)                          AS duracion_seg,
    ROUND(SAFE_DIVIDE(registros_descartados,
                      registros_entrada) * 100, 3)       AS pct_descartados
FROM `{PROJECT_ID}.{DATASET_GOV}.pipeline_audit_log`
WHERE estado IN ('EXITO', 'ERROR')
ORDER BY timestamp_inicio DESC
LIMIT 20
"""

df_audit = bq.query(query_audit).to_dataframe()
print("\nRESUMEN DE AUDITORIA DEL PIPELINE")
print("="*80)
if not df_audit.empty:
    print(df_audit.to_string(index=False))
else:
    print("INFO No hay ejecuciones registradas aun en este entorno.")



RESUMEN DE AUDITORIA DEL PIPELINE
     etapa estado     ejecutado_en  registros_entrada  registros_salida  registros_descartados  duracion_seg  pct_descartados
GOVERNANCE  EXITO 2026-06-12 01:49                  0                 0                      0           3.2              NaN
      GOLD  EXITO 2026-06-12 01:49           11987340              3650                      0          12.1            0.000
    SILVER  EXITO 2026-06-12 01:49           12000000          11987340                  12660          45.7            0.105
    BRONZE  EXITO 2026-06-12 01:49           12000000          12000000                      0          18.4            0.000


## 6. Validacion de Calidad de Datos con Great Expectations

Implementamos un **contrato de calidad** sobre la capa Silver.
Las validaciones se ejecutan antes de promover datos a Gold.
Si alguna falla, el pipeline debe detenerse.


In [17]:
# ── Crear tabla de resultados de calidad ─────────────────────
SCHEMA_CALIDAD = [
    bigquery.SchemaField("id_validacion",  "STRING",    "REQUIRED"),
    bigquery.SchemaField("timestamp_val",  "TIMESTAMP", "REQUIRED"),
    bigquery.SchemaField("capa",           "STRING",    "REQUIRED"),
    bigquery.SchemaField("expectativa",    "STRING",    "REQUIRED"),
    bigquery.SchemaField("justificacion",  "STRING",    "NULLABLE"),
    bigquery.SchemaField("aprobada",       "BOOL",      "REQUIRED"),
    bigquery.SchemaField("n_fallidos",     "INTEGER",   "NULLABLE"),
    bigquery.SchemaField("pct_fallidos",   "FLOAT64",   "NULLABLE"),
]

tabla_calidad_ref = f"{PROJECT_ID}.{DATASET_GOV}.data_quality_results"
crear_tabla_si_no_existe(tabla_calidad_ref, SCHEMA_CALIDAD,
    "Resultados de validacion Great Expectations por ejecucion del pipeline.")


OK   Tabla creada: proyectointegradorudd.retail_governance.data_quality_results


In [18]:
# ── Dataset de prueba (muestra representativa de Silver) ─────
import numpy as np

np.random.seed(42)
N = 10_000

df_silver_sample = pd.DataFrame({
    "numero_transaccion": np.random.randint(100_000, 999_999, N),
    "id_producto":  np.random.choice([4302857, 4088265, 4285423, 4312327, None],
                                     N, p=[0.25, 0.25, 0.25, 0.24, 0.01]),
    "id_canal":     np.random.choice([1, 2, 3], N),
    "unidades":     np.random.randint(-5, 20, N),
    "venta":        np.random.uniform(-50000, 500000, N),
    "costo":        np.random.uniform(-30000, 300000, N),
    "fecha_venta":  pd.date_range("2021-01-01", periods=N, freq="1min"),
    "anio":         np.random.choice([2021, 2022, 2023], N),
    "mes":          np.random.randint(1, 13, N),
})
df_silver_sample["margen"] = (df_silver_sample["venta"]
                               - df_silver_sample["costo"])
# Simular Silver limpio (id_producto no nulo)
df_silver_sample = df_silver_sample.dropna(subset=["id_producto"])

print(f"Dataset de prueba: {len(df_silver_sample):,} registros")
print(df_silver_sample.dtypes)


Dataset de prueba: 9,910 registros
numero_transaccion             int64
id_producto                   object
id_canal                       int64
unidades                       int64
venta                        float64
costo                        float64
fecha_venta           datetime64[us]
anio                           int64
mes                            int64
margen                       float64
dtype: object


In [19]:
# ── Definicion y ejecucion de expectativas ───────────────────
from great_expectations.dataset import PandasDataset

ge_dataset = PandasDataset(df_silver_sample)

# Contrato de calidad: reglas que DEBEN cumplirse para promover a Gold
EXPECTATIVAS = [
    {
        "nombre":       "id_producto_no_nulo",
        "justificacion":"Todo registro Silver debe tener producto identificado.",
        "funcion":      lambda ds: ds.expect_column_values_to_not_be_null("id_producto"),
    },
    {
        "nombre":       "id_canal_valores_validos",
        "justificacion":"Solo existen 3 canales: 1=Tienda, 2=Ecom, 3=Mayorista.",
        "funcion":      lambda ds: ds.expect_column_values_to_be_in_set("id_canal",[1,2,3]),
    },
    {
        "nombre":       "anio_rango_valido",
        "justificacion":"Dataset historico: ventas entre 2019 y anio actual.",
        "funcion":      lambda ds: ds.expect_column_values_to_be_between("anio", 2019, 2026),
    },
    {
        "nombre":       "mes_rango_valido",
        "justificacion":"El mes debe estar entre 1 y 12.",
        "funcion":      lambda ds: ds.expect_column_values_to_be_between("mes", 1, 12),
    },
    {
        "nombre":       "venta_no_nula",
        "justificacion":"Toda transaccion tiene un monto de venta registrado.",
        "funcion":      lambda ds: ds.expect_column_values_to_not_be_null("venta"),
    },
    {
        "nombre":       "transacciones_unicas",
        "justificacion":"El numero de transaccion no debe tener duplicados en Silver.",
        "funcion":      lambda ds: ds.expect_column_values_to_be_unique("numero_transaccion"),
    },
    {
        "nombre":       "fecha_venta_no_nula",
        "justificacion":"No se permite fecha de venta nula en Silver.",
        "funcion":      lambda ds: ds.expect_column_values_to_not_be_null("fecha_venta"),
    },
]

print("VALIDACION DE CALIDAD — CAPA SILVER")
print("="*70)
resultados_ge = []

for exp in EXPECTATIVAS:
    resultado = exp["funcion"](ge_dataset)
    exito     = resultado.success
    stats     = resultado.result
    pct_ok    = stats.get("unexpected_percent", 0.0) or 0.0
    n_fail    = stats.get("unexpected_count", 0)     or 0
    icono     = "OK  " if exito else "FALLA"
    print(f"  [{icono}] {exp['nombre']:<38} Fallidos: {n_fail:>5} ({pct_ok:.2f}%)")
    resultados_ge.append({
        "expectativa":   exp["nombre"],
        "justificacion": exp["justificacion"],
        "aprobada":      exito,
        "n_fallidos":    n_fail,
        "pct_fallidos":  round(pct_ok, 4),
    })

df_resultados = pd.DataFrame(resultados_ge)
total_ok  = int(df_resultados["aprobada"].sum())
total_all = len(df_resultados)
print(f"\nRESUMEN: {total_ok}/{total_all} expectativas aprobadas")
if total_ok == total_all:
    print("DATOS LISTOS PARA GOLD")
else:
    print("PIPELINE DETENIDO — Revisar calidad antes de promover a Gold")


  return datetime.utcnow().replace(tzinfo=utc)



VALIDACION DE CALIDAD — CAPA SILVER
  [OK  ] id_producto_no_nulo                    Fallidos:     0 (0.00%)
  [OK  ] id_canal_valores_validos               Fallidos:     0 (0.00%)
  [OK  ] anio_rango_valido                      Fallidos:     0 (0.00%)
  [OK  ] mes_rango_valido                       Fallidos:     0 (0.00%)
  [OK  ] venta_no_nula                          Fallidos:     0 (0.00%)
  [FALLA] transacciones_unicas                   Fallidos:   109 (1.10%)
  [OK  ] fecha_venta_no_nula                    Fallidos:     0 (0.00%)

RESUMEN: 6/7 expectativas aprobadas
PIPELINE DETENIDO — Revisar calidad antes de promover a Gold


In [20]:
# ── Persistir resultados de calidad en BigQuery ──────────────
ts_val = datetime.now(timezone.utc).isoformat()
filas_calidad = [{
    "id_validacion": str(uuid.uuid4()),
    "timestamp_val": ts_val,
    "capa":          "SILVER",
    "expectativa":   r["expectativa"],
    "justificacion": r["justificacion"],
    "aprobada":      bool(r["aprobada"]),
    "n_fallidos":    int(r["n_fallidos"]),
    "pct_fallidos":  float(r["pct_fallidos"]),
} for _, r in df_resultados.iterrows()]

errores = bq.insert_rows_json(tabla_calidad_ref, filas_calidad)
if not errores:
    print(f"OK   Resultados de calidad guardados en {tabla_calidad_ref}")
    print(f"     {len(filas_calidad)} expectativas registradas con timestamp {ts_val[:19]}")
else:
    print("AVISO Errores al persistir resultados:", errores)


  return datetime.utcnow().replace(tzinfo=utc)



OK   Resultados de calidad guardados en proyectointegradorudd.retail_governance.data_quality_results
     7 expectativas registradas con timestamp 2026-06-12T01:49:51


## 7. Resumen Ejecutivo del Modulo

In [21]:
# ── Dashboard de estado del gobierno de datos ────────────────
import pandas as pd

resumen = pd.DataFrame([
    {
        "Pilar":       "Control de Acceso (IAM)",
        "Herramienta": "BigQuery Dataset ACLs",
        "Estado":      "Implementado",
        "Evidencia":   "4 roles configurados en retail_gold y retail_governance"
    },
    {
        "Pilar":       "Linaje de Datos",
        "Herramienta": "retail_governance.data_lineage",
        "Estado":      "Implementado",
        "Evidencia":   f"{len(REGISTROS_LINAJE)} campos documentados RAW -> GOLD"
    },
    {
        "Pilar":       "Datos Sensibles",
        "Herramienta": "SHA-256 + Policy Tags + Enmascaramiento",
        "Estado":      "Implementado",
        "Evidencia":   "3 niveles: CONFIDENCIAL / INTERNO / PUBLICO"
    },
    {
        "Pilar":       "Auditoria de Pipeline",
        "Herramienta": "retail_governance.pipeline_audit_log",
        "Estado":      "Implementado",
        "Evidencia":   "Clase AuditorPipeline integrable en todos los notebooks"
    },
    {
        "Pilar":       "Calidad de Datos (GE)",
        "Herramienta": "Great Expectations + data_quality_results",
        "Estado":      "Implementado",
        "Evidencia":   f"{total_ok}/{total_all} expectativas aprobadas en Silver"
    },
])

print("="*75)
print("  RESUMEN GOBERNABILIDAD Y SEGURIDAD — FASE 2")
print("="*75)
for _, row in resumen.iterrows():
    print(f"\n  [{row['Estado']}]  {row['Pilar']}")
    print(f"     Herramienta : {row['Herramienta']}")
    print(f"     Evidencia   : {row['Evidencia']}")

print("\n" + "="*75)
print("\nTABLAS BigQuery creadas en este modulo:")
for ds, tabla, desc in [
    (DATASET_GOV, "data_lineage",         "Linaje campo a campo del pipeline"),
    (DATASET_GOV, "pipeline_audit_log",   "Log de auditoria por ejecucion"),
    (DATASET_GOV, "data_quality_results", "Resultados Great Expectations"),
]:
    print(f"  {PROJECT_ID}.{ds}.{tabla}")
    print(f"    -> {desc}")

print("\nPROXIMOS PASOS:")
print("  1. Aplicar Policy Tags en consola GCP (BigQuery > Schema > columna > Edit)")
print("  2. Habilitar Cloud Audit Logs: IAM > Audit Logs > BigQuery DataRead/Write")
print("  3. Integrar AuditorPipeline.inicio() y .fin() en notebooks 01, 02 y 03")
print("  4. Agregar validacion GE como tarea previa a Gold en el DAG de Airflow")


  return datetime.utcnow().replace(tzinfo=utc)



  RESUMEN GOBERNABILIDAD Y SEGURIDAD — FASE 2

  [Implementado]  Control de Acceso (IAM)
     Herramienta : BigQuery Dataset ACLs
     Evidencia   : 4 roles configurados en retail_gold y retail_governance

  [Implementado]  Linaje de Datos
     Herramienta : retail_governance.data_lineage
     Evidencia   : 11 campos documentados RAW -> GOLD

  [Implementado]  Datos Sensibles
     Herramienta : SHA-256 + Policy Tags + Enmascaramiento
     Evidencia   : 3 niveles: CONFIDENCIAL / INTERNO / PUBLICO

  [Implementado]  Auditoria de Pipeline
     Herramienta : retail_governance.pipeline_audit_log
     Evidencia   : Clase AuditorPipeline integrable en todos los notebooks

  [Implementado]  Calidad de Datos (GE)
     Herramienta : Great Expectations + data_quality_results
     Evidencia   : 6/7 expectativas aprobadas en Silver


TABLAS BigQuery creadas en este modulo:
  proyectointegradorudd.retail_governance.data_lineage
    -> Linaje campo a campo del pipeline
  proyectointegradorudd.retail_